# Bioserenity — all metrics (metadata + YASA sleep stats + infraslow)

For every subject that has **all three** inputs on `$OAK`

* metadata row in `Excel/bioserenity_metadata3.csv`,
* an EDF at `edf/{id}.edf`,
* a hypnodensity at `Sleep_Staging/{id}_Hypnodensity.csv`,

this notebook computes one row of metrics and exports a single merged CSV.

The heavy lifting lives in reusable functions in
[`infraslow.processing.all_metrics`](infraslow/processing/all_metrics.py); this
notebook just wires paths, calls them, and reports. It reproduces the
single-subject analysis of `demo_infraslow_yasa_average.ipynb` at cohort scale.

**Output columns** (52): `ID, Age, Gender, BMI` · the full YASA `sleep_statistics`
(`TIB, SPT, WASO, TST, N1, N2, N3, REM, NREM, SOL, Lat_*, %*, SE, SME`) · and for
each of `N2 / N3 / NREM`: `peak_freq(_sem)`, `fit_peak_freq(_sem)`, `bandwidth_hz`,
`auc`, `detected`, `bouts` (min), `%bouts` (of TST).

**Conventions** (see the module docstring for the full rationale):
* `peak_freq` = empirical argmax of the baseline-corrected relative spectrum;
  `fit_peak_freq` = Gaussian-fit centre. Both averaged across bouts; `_sem` is the
  standard error across bouts (`NaN` if <2 bouts).
* `bandwidth_hz`, `auc`, `detected` come from the fit of the bout-averaged
  spectrum (as in the reference notebook); `detected` = peak > 1.5·SD(baseline).
* `NREM` = N1+N2+N3 (matches the YASA `NREM` column); spindles for the bout filter
  are detected in N2+N3.
* `{stage}_bouts` = total minutes of consecutive stage runs ≥ 200 s; `%{stage}_bouts`
  = that as a percentage of **TST**.

> ⚠️ **Run this on a compute node** (`sh_dev` / `salloc` / `sbatch`), **not the
> login node**: it imports `lunapi` (EDF I/O) and `yasa`, and reads large EDFs.

## 1. Imports and configuration

In [7]:
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the src-layout `infraslow` package importable regardless of the kernel CWD.
def _find_up(marker: str) -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / marker).exists():
            return cand
    raise FileNotFoundError(f'Could not find {marker!r} above {here}')

REPO_ROOT = _find_up('pyproject.toml')
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from infraslow.io.hypnodensity import DEFAULT_HYPNODENSITY_SUFFIX
from infraslow.processing.all_metrics import (
    all_metric_columns,
    calculate_subject_all_metrics,
    find_valid_bioserenity_subjects,
    load_bioserenity_metadata,
    run_all_subjects,
    CHANNEL,
    SF,
    INFRASLOW_STAGES,
)

# INFO-level logs surface the per-subject OK/FAIL lines from run_all_subjects.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    force=True,
)
logging.getLogger('infraslow').setLevel(logging.INFO)
pd.set_option('display.max_columns', 100)

print(f'repo root : {REPO_ROOT}')
print(f'channel   : {CHANNEL} @ {SF:g} Hz   infraslow stages: {INFRASLOW_STAGES}')

repo root : /home/users/chaisaen/infraslow
channel   : C3 @ 128 Hz   infraslow stages: ('N2', 'N3', 'NREM')


## 2. Path setup using `$OAK`

In [8]:
oak_raw = os.environ.get('OAK')
if not oak_raw:
    raise EnvironmentError('$OAK is not set. Load your environment before running.')
OAK = Path(oak_raw)

BIOSERENITY = OAK / 'psg' / 'Bioserenity'
METADATA_PATH = BIOSERENITY / 'Excel' / 'Morpheus_Data_All5.csv'
EDF_DIR = BIOSERENITY / 'edf'
STAGING_DIR = BIOSERENITY / 'Sleep_Staging'

# The repo ships an `output/` folder; reuse it (the prompt's `outputs/` -> `output/`).
OUTPUT_DIR = REPO_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
FINAL_CSV = OUTPUT_DIR / 'all_metrics.csv'
FAILED_CSV = OUTPUT_DIR / 'all_metrics_failed_subjects.csv'

for label, path in [
    ('metadata', METADATA_PATH), ('edf dir', EDF_DIR), ('staging dir', STAGING_DIR),
]:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'{label:<12}: [{status}] {path}')
print(f'{"output":<12}: {OUTPUT_DIR}')

metadata    : [OK] /oak/stanford/groups/mignot/psg/Bioserenity/Excel/Morpheus_Data_All5.csv
edf dir     : [OK] /oak/stanford/groups/mignot/psg/Bioserenity/edf
staging dir : [OK] /oak/stanford/groups/mignot/psg/Bioserenity/Sleep_Staging
output      : /home/users/chaisaen/infraslow/output


## 3. Load metadata

Reads only `ID, Age, Gender, BMI` (the CSV is wide and long), tolerating minor
header-name differences.

In [9]:
metadata = load_bioserenity_metadata(METADATA_PATH)
print(f'metadata rows: {len(metadata):,}')
metadata.head()

metadata rows: 540,941


,ID,Age,Gender,BMI
0,8631,57.0,Male,40.0
1,8626,52.0,Female,47.0
2,10516,59.0,Male,32.0
3,7438,68.0,Female,31.0
4,7515,58.0,Male,50.0


## 4. Find valid subjects (matching EDF + hypnodensity)

Lists each directory once (single `readdir`) and tests membership in memory —
no per-subject `stat` storm against Lustre.

In [10]:
valid = find_valid_bioserenity_subjects(metadata, EDF_DIR, STAGING_DIR)
availability = valid.attrs['availability']
print('availability:')
for key, value in availability.items():
    print(f'  {key:<22}: {value:,}')
valid.head()

availability:
  n_metadata            : 540,941
  n_with_edf            : 150,626
  n_with_hypnodensity   : 143,847
  n_valid               : 143,413


,ID,Age,Gender,BMI
0,335879,54.0,Female,46.0
1,336094,56.0,Male,32.0
2,320244,57.0,Female,33.0
3,330396,39.0,Female,46.0
4,337020,73.0,Female,58.0


## 5. Helper functions (imported from `infraslow.processing.all_metrics`)

The pipeline is built from these reusable functions (all documented in the module):

| Function | Role |
|---|---|
| `load_bioserenity_metadata` | metadata → standardized `ID, Age, Gender, BMI` |
| `find_valid_bioserenity_subjects` | keep subjects with both EDF + hypnodensity |
| `load_hypnodensity_as_hypnogram` | hypnodensity CSV → YASA integer hypnogram (argmax) |
| `calculate_yasa_sleep_statistics` | full YASA `sleep_statistics` |
| `calculate_stage_bouts` | consolidated-bout duration per stage |
| `calculate_stage_infraslow` | infraslow peak/bandwidth/AUC/detection per stage |
| `calculate_subject_infraslow_metrics` | all stage-groups for one subject |
| `calculate_subject_all_metrics` | metadata + YASA + infraslow for one subject |
| `run_all_subjects` | cohort runner with per-subject error isolation |

## 6. Test the full pipeline on one subject

Sanity-check the end-to-end computation on the first valid subject before the
full run. (This reads one EDF — expect it to take a little while.)

In [11]:
if valid.empty:
    raise RuntimeError('No valid subjects found — check the $OAK paths above.')

test_row = valid.iloc[0]
test_id = test_row['ID']
test_edf = EDF_DIR / f'{test_id}.edf'
test_hypno = STAGING_DIR / f'{test_id}{DEFAULT_HYPNODENSITY_SUFFIX}'
print(f'testing subject: {test_id}')

one_row = calculate_subject_all_metrics(test_row, test_edf, test_hypno)
pd.Series(one_row)[all_metric_columns()].to_frame(name=test_id)

2026-07-09 14:33:38,403 INFO infraslow.io.psg_loader: Loading subject 335879 from /oak/stanford/groups/mignot/psg/Bioserenity/edf/335879.edf
  *** warning, signal 18 ( SAO2 ) has a non-integer SR
  *** may wish to use RECORD-SIZE if this is due to a fractional EDF record size
  *** (it will resample signals to fit in a new regular, e.g. dur=1, record size)
    - record size        = 2 seconds
    - samples per record = 25
    - implied SR         = 25 / 2 = 12.5
  *** warning, signal 19 ( IPAP ) has a non-integer SR
  *** may wish to use RECORD-SIZE if this is due to a fractional EDF record size
  *** (it will resample signals to fit in a new regular, e.g. dur=1, record size)
    - record size        = 2 seconds
    - samples per record = 25
    - implied SR         = 25 / 2 = 12.5
  *** warning, signal 20 ( EPAP ) has a non-integer SR
  *** may wish to use RECORD-SIZE if this is due to a fractional EDF record size
  *** (it will resample signals to fit in a new regular, e.g. dur=1, re

testing subject: 335879
initiated lunapi 1.4.0 <lunapi.lunapi0.luna object at 0x7f96a61045f0> 



2026-07-09 14:33:40,089 INFO infraslow.io.psg_loader: Loaded 1 channel(s), 3352320 sample(s); 0 missing, 0 conflict record(s).
2026-07-09 14:33:41,258 WARNING yasa: Hypnogram is SHORTER than data by 30.00 seconds. Padding hypnogram with Unscored (UNS) to match data.size.


Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 423 samples (3.305 s)

Setting up band-pass filter from 12 - 15 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 12.00
- Lower transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 11.25 Hz)
- Upper passband edge: 15.00 Hz
- Upper transition bandwidth: 1.50 Hz (-6 dB cutoff frequency: 15.75 Hz)
- F

,335879
ID,335879
Age,54.0
Gender,Female
BMI,46.0
TIB,436.0
SPT,371.0
WASO,67.5
TST,303.5
N1,20.0
N2,268.0


## 7. Run the pipeline for all valid subjects

Set `MAX_SUBJECTS` to a small integer for a quick smoke test; leave it `None` to
process the **entire** valid cohort. Each subject is isolated in a `try/except`,
so a single bad recording is recorded and skipped rather than aborting the run.
For the full cohort, submit this notebook as a batch job (`papermill` / `jupyter
nbconvert --execute`) with enough walltime and memory.

In [12]:
MAX_SUBJECTS = None  # e.g. 10 for a quick test; None = all valid subjects

subset = valid if MAX_SUBJECTS is None else valid.head(MAX_SUBJECTS)
print(f'processing {len(subset):,} subject(s)...')

results_df, failed_df = run_all_subjects(subset, EDF_DIR, STAGING_DIR)
print(f'\nprocessed OK : {len(results_df):,}')
print(f'failed       : {len(failed_df):,}')

processing 143,413 subject(s)...


Subjects:   0%|          | 0/143413 [00:00<?, ?subj/s]2026-07-09 14:33:56,637 INFO infraslow.io.psg_loader: Loading subject 335879 from /oak/stanford/groups/mignot/psg/Bioserenity/edf/335879.edf
  *** warning, signal 18 ( SAO2 ) has a non-integer SR
  *** may wish to use RECORD-SIZE if this is due to a fractional EDF record size
  *** (it will resample signals to fit in a new regular, e.g. dur=1, record size)
    - record size        = 2 seconds
    - samples per record = 25
    - implied SR         = 25 / 2 = 12.5
  *** warning, signal 19 ( IPAP ) has a non-integer SR
  *** may wish to use RECORD-SIZE if this is due to a fractional EDF record size
  *** (it will resample signals to fit in a new regular, e.g. dur=1, record size)
    - record size        = 2 seconds
    - samples per record = 25
    - implied SR         = 25 / 2 = 12.5
  *** warning, signal 20 ( EPAP ) has a non-integer SR
  *** may wish to use RECORD-SIZE if this is due to a fractional EDF record size
  *** (it will re

initiated lunapi 1.4.0 <lunapi.lunapi0.luna object at 0x7f96a61045f0> 



Subjects:   0%|          | 0/143413 [00:00<?, ?subj/s]


KeyboardInterrupt: 

## 8. Save the final CSV(s)

In [ ]:
results_df.to_csv(FINAL_CSV, index=False)
failed_df.to_csv(FAILED_CSV, index=False)
print(f'metrics       -> {FINAL_CSV}')
print(f'failed subjects -> {FAILED_CSV}')

## 9. Processing summary

In [ ]:
print('=' * 60)
print('PROCESSING SUMMARY')
print('=' * 60)
print(f'Metadata subjects          : {availability["n_metadata"]:,}')
print(f'  with EDF file            : {availability["n_with_edf"]:,}')
print(f'  with hypnodensity file   : {availability["n_with_hypnodensity"]:,}')
print(f'Valid (both files present) : {availability["n_valid"]:,}')
print(f'Attempted this run         : {len(subset):,}')
print(f'Successfully processed     : {len(results_df):,}')
print(f'Failed                     : {len(failed_df):,}')
print('-' * 60)
if not failed_df.empty:
    print('Failures by error_type:')
    for etype, count in failed_df['error_type'].value_counts().items():
        print(f'  {etype:<28}: {count}')
    print('-' * 60)
print(f'Final metrics CSV : {FINAL_CSV}')
print(f'Failed subjects   : {FAILED_CSV}')
print('=' * 60)

## 10. Sanity checks: `head()` and `describe()`

In [ ]:
results_df.head()

In [ ]:
results_df.describe(include='all').T

In [ ]:
# Detection rate per stage (quick quality read-out).
for stage in INFRASLOW_STAGES:
    col = f'{stage}_detected'
    if col in results_df.columns and len(results_df):
        rate = results_df[col].mean()
        print(f'{stage:<5} infraslow detected: {rate:6.1%}  ({int(results_df[col].sum())}/{len(results_df)})')